In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 11 — Analisis PU (eksploratif)

**Tujuan:** menguji apakah formulasi positive-unlabeled memperbaiki peringkat dibanding model naive.

`s` adalah label teramati (hilang pada snapshot). `y` laten berarti akun kelak hilang kapan pun.
Bila `y` dimaknai sebagai akun bermasalah, asumsi p(s=1|y=0)=0 bisa gagal, karena akun bisa dihapus sendiri.
Semua hasil di notebook ini bersifat eksploratif.

In [3]:
import json

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats as scipy_stats
from sklearn.metrics import average_precision_score

from ghremoval.analysis import score_row
from ghremoval.config import load_context
from ghremoval.data import xy
from ghremoval.models.fit import arm_mask, load_stage, predict
from ghremoval.pu import bagging_score, elkan_noto_fit, hide_positives, naive_fit, pu_bagging_fit, sar_em_fit
from ghremoval.quality import Checks
from ghremoval.report import save_table
from ghremoval.runlog import RunLog

ctx = load_context(config_path)
run = RunLog("11_pu_analysis", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
data = load_stage(ctx, "dev", 60)
X, s = xy(data.train, "M_behav")
X_test, y_test = xy(data.test, "M_behav")
propensity = {"activity": data.train.select("n_events_coarse").to_numpy(), "age": data.train.select("age_bin12").to_numpy(), "activity+age": data.train.select("n_events_coarse", "age_bin12").to_numpy()}
mask = arm_mask(data.train, data.test, "F2_forward_new_accounts", ctx)
strata = data.test["stratum"].to_numpy()
SEED = ctx.p.project["seed"]

## 1. Metode pada label teramati

In [4]:
naive = naive_fit(X, s, SEED)
en_model, c_mean = elkan_noto_fit(X, s, SEED)
bags = pu_bagging_fit(X, s, SEED)
sar = {name: sar_em_fit(X, s, features, SEED)[0] for name, features in propensity.items()}
methods = {"naive": predict(naive, X_test), "elkan_noto": predict(en_model, X_test), "pu_bagging": bagging_score(bags, X_test), **{f"sar_em_{name}": predict(model, X_test) for name, model in sar.items()}}
observed = pd.DataFrame([{"method": name, **score_row(y_test, score, strata, mask)} for name, score in methods.items()])
observed[["method", "n", "n_pos", "ap_all", "ap_5-19", "ap_20+", "p_at_100"]]

,method,n,n_pos,ap_all,ap_5-19,ap_20+,p_at_100
0,naive,22095,560,0.207501,0.178884,0.509065,0.57
1,elkan_noto,22095,560,0.215785,0.185540,0.540084,0.60
2,pu_bagging,22095,560,0.184609,0.156735,0.490479,0.47
3,sar_em_activity,22095,560,0.205515,0.175310,0.530513,0.58
4,sar_em_age,22095,560,0.202247,0.170700,0.529353,0.54
5,sar_em_activity+age,22095,560,0.205308,0.173760,0.526487,0.57


## 2. Prior per stratum, sebagai rentang

`c` diperkirakan dari positif pada data latih yang disisihkan. Rentang datang dari beberapa estimator sederhana.
Karena pustaka estimator prior khusus tidak dipasang, rentang ini hanya memberi gambaran kasar.

In [5]:
rng = np.random.default_rng(SEED)
held = rng.random(len(s)) < 0.2
clf = naive_fit(X[~held], s[~held], SEED)
held_positive_scores = predict(clf, X[held & (s == 1)])
estimators = {"c_mean": float(held_positive_scores.mean()), "c_median": float(np.median(held_positive_scores)), "c_q90": float(np.quantile(held_positive_scores, 0.9))}
train_strata = data.train["stratum"].to_numpy()
prior_rows = []
for stratum in ("5-19", "20+"):
    observed_rate = float(s[train_strata == stratum].mean())
    prior_rows.append({"stratum": stratum, "observed_s_rate": observed_rate, **{f"prior_{k}": min(1.0, observed_rate / v) for k, v in estimators.items()}})
priors = pd.DataFrame(prior_rows)
print(estimators, "Elkan-Noto c:", round(c_mean, 4))
priors

{'c_mean': 0.3631337881088257, 'c_median': 0.07269042730331421, 'c_q90': 0.9956385493278503} Elkan-Noto c: 0.3553


,stratum,observed_s_rate,prior_c_mean,prior_c_median,prior_c_q90
0,5-19,0.027906,0.076847,0.383898,0.028028
1,20+,0.074349,0.204744,1.000000,0.074675


## 3. Uji semi-sintetis

50% positif train disembunyikan, peluangnya bergantung pada aktivitas dan usia. Tiap metode dilatih pada label yang sudah disembunyikan.

In [6]:
def recovery_ap(score: np.ndarray, hidden: np.ndarray, s_hidden: np.ndarray) -> float:
    unlabeled = s_hidden == 0
    return float(average_precision_score(hidden[unlabeled], score[unlabeled]))


records = []
for repeat in range(5):
    hidden = hide_positives(s, propensity["activity+age"], 0.5, SEED + repeat)
    s_hidden = s & ~hidden
    scores = {"naive": predict(naive_fit(X, s_hidden, SEED + repeat), X), "elkan_noto": predict(elkan_noto_fit(X, s_hidden, SEED + repeat)[0], X), "pu_bagging": bagging_score(pu_bagging_fit(X, s_hidden, SEED + repeat), X)}
    for name, features in propensity.items():
        scores[f"sar_em_{name}"] = predict(sar_em_fit(X, s_hidden, features, SEED + repeat)[0], X)
    records.append({name: recovery_ap(score, hidden, s_hidden) for name, score in scores.items()})
semi = pd.DataFrame(records)
print(semi.mean().round(4).to_dict())
best_pu = semi.drop(columns=["naive"]).mean().idxmax()
delta = semi[best_pu] - semi["naive"]
t = scipy_stats.t.ppf(0.975, len(delta) - 1)
ci = (float(delta.mean() - t * delta.std(ddof=1) / np.sqrt(len(delta))), float(delta.mean() + t * delta.std(ddof=1) / np.sqrt(len(delta))))
print("metode PU terbaik:", best_pu, "selisih rata-rata:", round(delta.mean(), 4), "CI 95%:", np.round(ci, 4))

{'naive': 0.3614, 'elkan_noto': 0.3571, 'pu_bagging': 0.3262, 'sar_em_activity': 0.3628, 'sar_em_age': 0.3714, 'sar_em_activity+age': 0.3669}
metode PU terbaik: sar_em_age selisih rata-rata: 0.01 CI 95%: [0.0053 0.0147]


## Ringkasan & cek

In [7]:
semi_summary = semi.agg(["mean", "std"]).T.reset_index(names="method")
report = observed.merge(semi_summary.rename(columns={"mean": "semi_synth_ap_mean", "std": "semi_synth_ap_std"}), on="method", how="left")
save_table(report, "tab_pu_methods", "Exploratory PU analysis (observed label and semi-synthetic recovery)")
save_table(priors, "tab_pu_priors", "Exploratory class-prior ranges")
checks.add("Q17", "w3" not in set(data.test["week"].unique()), "tanpa w3")
checks.raise_if_failed()
h3 = {"best_pu": best_pu, "delta": float(delta.mean()), "ci": ci, "h3_supported": bool(ci[0] > 0)}
(ctx.paths.results / "pu_dev.json").write_text(json.dumps(h3, indent=2) + "\n")
print(h3)
print(run.finish(n_train=len(s)))

[LULUS] Q17: tanpa w3
{'best_pu': 'sar_em_age', 'delta': 0.009998459374373603, 'ci': (0.005251520648054706, 0.0147453981006925), 'h3_supported': True}
{'notebook': '11_pu_analysis', 'finished_at_utc': '2026-11-10T11:57:43+00:00', 'git_commit': 'f04dbd3', 'duration_seconds': 105.4, 'counts': {'n_train': 55439}}
